### STA-LTA Trigger

This notebook illustrates how the STA-LTA trigger can be used to extract waveform segments from seismic data. 

For the illustration we consider data from ILL18 for 2018. We note the following:

1. Data should be contained in the folder "../data/XP/2018/ILL18/EHZ.D".
2. Output is stored in the folder "../output/illustration"
3. The debris-flow catalog is contained in "../catalogs/XP/flow_catalog.csv". 

In [3]:
# imports
import os
import numpy as np
import obspy
import pandas as pd

from seismicif.datamod.loading_utils import (
    preproc_flow_annotations,
    find_date_in_strings,
    remove_duplicate_traces,
    remove_overlaps,
    extract_split_flows,
)

from seismicif.datamod.preproc_utils import preproc_stream
from seismicif.sta_lta import slta_compute_iou, slta_detections_from_paths
from seismicif.metrics import (
    iou,
    est_thresholds,
    compute_statistics,
    extract_valid_segments,
)
from datetime import timedelta
from tqdm import tqdm
from itertools import product

In [6]:
# define paths to folder
folder = "../data/XP/2018/ILL18/EHZ.D/"
stream_paths = [folder + f for f in os.listdir(folder) if not f.startswith("._")]
stream_paths = np.array(sorted(stream_paths, key=lambda f: int(f.rsplit(".", 1)[-1])))

# read catalog
flows = preproc_flow_annotations(pd.read_csv("../catalogs/XP/flow_catalog.csv"))
lower_conf_flows, high_conf_flows, all_flows = extract_split_flows(
    flows, "ILL18", 2018, 2018
)

In constrast to the IF trigger we found it difficult to find good generalizable rule of thumb values and so we skip directly to calibration. We calibrate the STA-LTA trigger by focusing only on days with catalog segments, regardless of confidence levels, as we did for the IF trigger.

Note. As far as is feasible (i.e. available data and no breakages) we append the seismic waveform of the previous day so that the STA-LTA can immediately compute its characteristic function.  

In [7]:
# extract and append to recordings
dates = []
for i in range(all_flows.shape[0]):
    dates.append(all_flows["start"].iloc[i].date)
    dates.append(all_flows["stop"].iloc[i].date)

dates = np.array(dates)
dates = np.unique(dates)

prev_days = []

for dt in dates:
    prev_days.append(dt - timedelta(days=1))

prev_days = np.array(prev_days)

st = []

for i in tqdm(range(dates.shape[0])):
    flow_st = obspy.read(find_date_in_strings(stream_paths, dates[i])[0])
    flow_st = remove_duplicate_traces(flow_st)
    flow_st = remove_overlaps(flow_st)
    preproc_stream(flow_st)

    prev_day_st = obspy.read(find_date_in_strings(stream_paths, prev_days[i])[0])
    prev_day_st = remove_duplicate_traces(prev_day_st)
    prev_day_st = remove_overlaps(prev_day_st)
    preproc_stream(prev_day_st)

    st.append({"res_tr": prev_day_st[-1], "st": flow_st})

100%|██████████| 7/7 [00:06<00:00,  1.15it/s]


In [9]:
# initilaize grids
sampling_rate = 100
lw_grid = 5000 * np.power(2.0, np.arange(-5, 6))
sw_grid = 0.003125 * np.power(2.0, np.arange(11))
onset_grid = 0.1875 * np.power(2.0, np.arange(11))
offset_grid = 0.00390625 * np.power(2.0, np.arange(11))

# perform local grid searches until no improvement in the IoU can be obtained.
# iou_table is a bit artificial, but the grid search never yielded a parameter on the boundary.
try:
    iou_table = np.load("../output/illustration/ILL18_slta_iou_tab.npy")

except (FileNotFoundError, OSError):
    iou_table = np.zeros([11, 11, 11, 11])
    iou_table[:] = np.nan
    ic, jc, kc, lc = 5, 5, 5, 5
    stop = False

    while not stop:
        print("Performing Local Search")
        stop = True

        i_range = range(max(0, ic - 1), min(10, ic + 2))
        j_range = range(max(0, jc - 1), min(10, jc + 2))
        k_range = range(max(0, kc - 1), min(10, kc + 2))
        l_range = range(max(0, lc - 1), min(10, lc + 2))

        for i, j, k, l in tqdm(
            product(i_range, j_range, k_range, l_range),
            total=len(i_range) * len(j_range) * len(k_range) * len(l_range),
        ):
            if not np.isnan(iou_table[i, j, k, l]):
                continue

            stop = False
            lw = int(sampling_rate * lw_grid[j])
            sw = int(sampling_rate * sw_grid[i] * lw_grid[j])
            onset_thres, offset_thres = onset_grid[k], offset_grid[l]

            iou_table[i, j, k, l] = slta_compute_iou(
                st, all_flows, sw, lw, onset_thres, offset_thres
            )

        # all the positions around the current one has been filled.
        if stop:
            break

        temp_table = iou_table.copy()
        temp_table[np.isnan(temp_table)] = 0

        # check if we found a better solution than the current...otherwise stop
        if np.max(temp_table) > iou_table[ic, jc, kc, lc]:
            stop = False
            ic, jc, kc, lc = np.unravel_index(np.argmax(temp_table), temp_table.shape)

        else:
            stop = True

        np.save(f"../output/illustration/ILL18_slta_iou_tab.npy", iou_table)

iou_table[np.isnan(iou_table)] = 0
iou_table[np.isnan(iou_table)] = 0
ic, jc, kc, lc = np.unravel_index(np.argmax(iou_table), iou_table.shape)

lw = int(sampling_rate * lw_grid[jc])
sw = int(sampling_rate * sw_grid[ic] * lw_grid[jc])
onset_thres, offset_thres = onset_grid[kc], offset_grid[lc]
print(f"max iou: {100 * iou_table[ic, jc, kc, lc]:.2f}%")
print(
    f"Best parameters: lw={lw}, sw={sw}, onset_thres={onset_thres}, offset_thres={offset_thres}"
)

Performing Local Search


100%|██████████| 81/81 [11:38<00:00,  8.62s/it]


Performing Local Search


100%|██████████| 81/81 [06:53<00:00,  5.10s/it]


Performing Local Search


100%|██████████| 81/81 [10:31<00:00,  7.80s/it]


Performing Local Search


100%|██████████| 81/81 [06:26<00:00,  4.77s/it]

max iou: 45.09%
Best parameters: lw=4000000, sw=200000, onset_thres=12.0, offset_thres=0.5


Extract segments using calibrated hyper parameters.

In [10]:
# extract segments
sta_lta_segments = slta_detections_from_paths(
    stream_paths, sw, lw, onset_thres, offset_thres
)

sta_lta_segments.to_csv("../output/illustration/ILL18_slta_segments.csv")
sta_lta_segments.head()

100%|██████████| 181/181 [06:46<00:00,  2.25s/it]


,start,stop,scores
0,2018-05-31T08:01:53.010003Z,2018-05-31T10:54:34.640003Z,19.890118
1,2018-10-25T22:58:12.200000Z,2018-10-25T23:36:35.180000Z,19.814092
2,2018-08-08T17:45:41.480002Z,2018-08-08T18:26:13.100002Z,19.776783
3,2018-08-09T14:38:24.280002Z,2018-08-09T16:10:01.120002Z,19.738901
4,2018-07-25T16:52:23.460019Z,2018-07-25T17:54:14.110019Z,19.671926
